# Efficient TS batching via PackedSequence


<https://discuss.pytorch.org/t/customized-rnn-cell-which-can-accept-packsequence/1067>

- https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.PackedSequence.html


In [1]:
import time
import numpy as np
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_sequence, pad_sequence, pack_padded_sequence, pad_packed_sequence, PackedSequence

device = torch.device("cuda")
dtype = torch.float32

In [2]:
def pack(sequence: list[torch.Tensor], **kwargs) -> tuple[PackedSequence, list[int]]:
    lengths = list(map(len, sequence))
    tensors = [tensor for length, tensor in zip(lengths, sequence) if length > 0]
    packed_sequence = pack_sequence(tensors, **kwargs)
    return packed_sequence, lengths


def unpack(packed_sequence: PackedSequence, lengths: list[int]) -> list[torch.Tensor]:
    device = packed_sequence.data.device
    dtype = packed_sequence.data.dtype
    trailing_dims = packed_sequence.data.shape[1:]
    unpacked_sequence = []
    idx_map = {}
    head = 0
    for b_idx, length in enumerate(lengths):
        unpacked_sequence.append(torch.zeros(length, *trailing_dims, device=device, dtype=dtype))
        if length > 0:
            idx_map[head] = b_idx
            head += 1
    head = 0
    for l_idx, b_size in enumerate(packed_sequence.batch_sizes):
        for b_idx in range(b_size):
            unpacked_sequence[idx_map[b_idx]][l_idx] = packed_sequence.data[head]
            head += 1
    return unpacked_sequence

In [3]:
# model creation
batch_size = 32
input_size = 100
hidden_size = 512
seq_len_range = (10, 1000)
num_batches = 10

rnn = nn.RNN(input_size, hidden_size, num_layers=4, bias=True, batch_first=True)
rnn.to(device)
rnn.zero_grad()

In [4]:
# data generation
batches = list()
for idx in range(num_batches):
    batch = []
    for k in range(batch_size):
        rand_len = np.random.randint(*seq_len_range)
        x = torch.rand((rand_len, input_size), device=device)
        y = torch.rand((rand_len, hidden_size), device=device)
        batch += [(x,y)]
    batch = sorted(batch, key=lambda x: x[0].size(0), reverse=True)
    batches += [batch]

In [5]:
# for padded input
start = time.time()
for batch in batches:
    yhat = []
    l = torch.tensor(0, dtype=dtype, device=device)
    for x, y in batch:
        yhat = rnn(x.unsqueeze(0))[0].squeeze(dim=0)
        r = (y-yhat)**2
        l += torch.sum(r)
    l.backward()
    g = torch.cat([w.grad.flatten() for w in rnn.parameters()])
    rnn.zero_grad()
end = time.time()
print(f"elapsed time for padded input: {end - start} secs")
print(torch.sum(torch.isnan(g)))
print(r)

elapsed time for padded input: 9.714176416397095 secs
tensor(0, device='cuda:0')
tensor([[2.8392e-01, 2.9016e-01, 8.7988e-01,  ..., 6.2053e-01, 1.3184e-02,
         7.4646e-02],
        [1.1943e+00, 4.9929e-01, 3.0616e-02,  ..., 1.5069e-02, 1.7605e-01,
         1.4230e-03],
        [1.9320e-01, 6.8433e-02, 1.8208e-01,  ..., 7.5675e-01, 8.8446e-04,
         1.4113e-01],
        ...,
        [2.7825e-01, 9.2636e-02, 5.7949e-02,  ..., 1.0080e+00, 9.7776e-02,
         4.8691e-05],
        [1.2005e+00, 6.1304e-01, 2.5057e-02,  ..., 8.0446e-01, 2.8204e-02,
         4.6612e-01],
        [1.6491e-01, 1.0497e-01, 4.0407e-01,  ..., 1.6664e-01, 1.5183e-02,
         3.0235e-01]], device='cuda:0', grad_fn=<PowBackward0>)


In [6]:
# for padded input
start = time.time()
for batch in batches:
    x, y = zip(*batch)
    x = pad_sequence(x, padding_value=np.nan, batch_first=True)
    y = pad_sequence(y, padding_value=np.nan, batch_first=True)
    yhat = rnn(x)[0]
    mask = torch.isnan(yhat)
    zero = torch.tensor(0, dtype=dtype, device=device)
    r = torch.where(mask, zero, (y-yhat)**2 )
    l = torch.sum(r)
    l.backward()
    g = torch.cat([w.grad.flatten() for w in rnn.parameters()])
    rnn.zero_grad()
end = time.time()
print(f"elapsed time for padded input: {end - start} secs")
print(torch.sum(torch.isnan(g)))
print(r.flatten())

elapsed time for padded input: 1.2893128395080566 secs
tensor(1890304, device='cuda:0')
tensor([0.3477, 0.0477, 0.0427,  ..., 0.0000, 0.0000, 0.0000], device='cuda:0',
       grad_fn=<ViewBackward>)


In [7]:
# for packed input
start = time.time()
for batch in batches:
    x, y = zip(*batch)
    x = pack_sequence(x)
    y = pack_sequence(y)
    yhat = rnn(x)[0]
    r = (y.data-yhat.data)**2
    l = torch.sum(r)
    l.backward()
    g = torch.cat([w.grad.flatten() for w in rnn.parameters()])
    rnn.zero_grad()
end = time.time()
print(f"elapsed time for packed input: {end - start} secs")
print(torch.sum(torch.isnan(g)))
print(r)

elapsed time for packed input: 1.174264907836914 secs
tensor(0, device='cuda:0')
tensor([[0.3477, 0.0477, 0.0427,  ..., 0.5496, 0.0431, 0.1010],
        [0.5443, 0.6806, 0.0681,  ..., 0.1767, 0.3196, 0.0694],
        [0.0444, 0.5757, 0.0535,  ..., 0.0278, 0.0139, 0.0932],
        ...,
        [0.1606, 0.0064, 0.0190,  ..., 0.4349, 0.3051, 0.4936],
        [0.4823, 0.6918, 0.4796,  ..., 0.1930, 0.6572, 0.0011],
        [0.4637, 0.4374, 0.1695,  ..., 0.4318, 0.1028, 0.0565]],
       device='cuda:0', grad_fn=<PowBackward0>)


In [8]:
# for packed input with unpack
start = time.time()
for batch in batches:
    x_batch, y_batch  = zip(*batch)
    x_packed, _       = pack(x_batch)
    y_packed, lengths = pack(y_batch)
    yhat_packed = rnn(x_packed)[0]
    
    r = torch.tensor(0, dtype=dtype, device=device)
    for y, yhat in zip(y_batch, unpack(y_packed, lengths)):
        r += torch.mean( (y-yhat)**2 )
    r.backward()
    g = torch.cat([w.grad.flatten() for w in rnn.parameters()])
    print(torch.sum(torch.isnan(g)))
    rnn.zero_grad()
end = time.time()
print(f"elapsed time for packed input: {end - start} secs")

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [9]:
dtype=torch.float32
device=torch.device('cpu')
rnn = nn.RNN(2, 2, num_layers=4, bias=True, batch_first=True)
rnn.to(device)


RNN(2, 2, num_layers=4, batch_first=True)

In [10]:
a = torch.tensor(np.random.randint(0, 9, (5, 2)), dtype=dtype, device=device)
b = torch.tensor(np.random.randint(0, 9, (4, 2)), dtype=dtype, device=device)
c = torch.tensor(np.random.randint(0, 9, (3, 2)), dtype=dtype, device=device)

In [11]:
batch = [a,b,c]
lengths = [len(x) for x in batch]
x, lengths = pack([a,b,c])
rnn(x)

(PackedSequence(data=tensor([[0.7213, 0.5935],
         [0.7254, 0.6163],
         [0.7213, 0.5935],
         [0.7289, 0.7626],
         [0.7317, 0.7402],
         [0.7279, 0.7585],
         [0.7597, 0.7461],
         [0.7564, 0.7860],
         [0.7640, 0.7741],
         [0.7411, 0.8012],
         [0.7592, 0.7802],
         [0.7715, 0.7863]], grad_fn=<CatBackward>), batch_sizes=tensor([3, 3, 3, 2, 1]), sorted_indices=None, unsorted_indices=None),
 tensor([[[ 0.9947,  0.5827],
          [ 0.9999,  0.5121],
          [ 0.9993,  0.7094]],
 
         [[ 0.4950,  0.6492],
          [ 0.5164,  0.6091],
          [ 0.5318,  0.5942]],
 
         [[-0.2149,  0.4878],
          [-0.1913,  0.4530],
          [-0.1995,  0.4778]],
 
         [[ 0.7715,  0.7863],
          [ 0.7592,  0.7802],
          [ 0.7640,  0.7741]]], grad_fn=<StackBackward>))

In [12]:
y = rnn(x)[0]
y = unpack(y, lengths)
yhat = [rnn(z.unsqueeze(dim=0))[0] for z in batch]
[z-zhat for z,zhat in zip(y,yhat)]

[tensor([[[0., 0.],
          [0., 0.],
          [0., 0.],
          [0., 0.],
          [0., 0.]]], grad_fn=<SubBackward0>),
 tensor([[[0., 0.],
          [0., 0.],
          [0., 0.],
          [0., 0.]]], grad_fn=<SubBackward0>),
 tensor([[[0., 0.],
          [0., 0.],
          [0., 0.]]], grad_fn=<SubBackward0>)]

In [13]:
batch = pad_sequence(batch, padding_value=np.nan, batch_first=True)
rnn(batch)

(tensor([[[0.7213, 0.5935],
          [0.7289, 0.7626],
          [0.7597, 0.7461],
          [0.7411, 0.8012],
          [0.7715, 0.7863]],
 
         [[0.7254, 0.6163],
          [0.7317, 0.7402],
          [0.7564, 0.7860],
          [0.7592, 0.7802],
          [   nan,    nan]],
 
         [[0.7213, 0.5935],
          [0.7279, 0.7585],
          [0.7640, 0.7741],
          [   nan,    nan],
          [   nan,    nan]]], grad_fn=<TransposeBackward1>),
 tensor([[[ 0.9947,  0.5827],
          [    nan,     nan],
          [    nan,     nan]],
 
         [[ 0.4950,  0.6492],
          [    nan,     nan],
          [    nan,     nan]],
 
         [[-0.2149,  0.4878],
          [    nan,     nan],
          [    nan,     nan]],
 
         [[ 0.7715,  0.7863],
          [    nan,     nan],
          [    nan,     nan]]], grad_fn=<StackBackward>))